### RaschPy Differential Item Functioning (DIF) worked example

Differential Item Functioning occurs when persons with the same overall trait level, but from different groups (e.g. Gender, age or first language), have different probabilities of success on a given item, which violates measurement principles and represents a threat to the validity of scores if unaddressed. `dif_test()` (available for `SLM`, `PCM`, `RSM` and `MFRM`; `MFRM` also has `drf_test()` for differential rater functioning) tests every item for DIF against a chosen reference group:

1. Reads in a file mapping persons to exogenous variables, stored as an attribute `self.exogenous`. This file does not need to have a mapping for every person to every variable; the test ignores persons who are unlabelled for the relevant variable.
2. Splits non-extreme persons into groups by a named column in `self.exogenous` (any number of groups; each "focal" group is compared to a single "reference" group, by default the largest).
3. Calibrates each group independently, on the items its persons responded to and with the object's `calibrate` settings (for `MFRM`, when the groups are scored by different facet elements, each group is anchored on the facet elements shared by both groups), then purifies both groups onto a common scale — genuinely DIF-affected items are excluded from the scale used to detect DIF so they do not distort the test.
4. Tests every item (not only the ones used to define the purified scale) via a per-item Wald test, likelihood-ratio test, or both, plus an omnibus test that tests whether DIF exists in any items in the data. $ p $-values are corrected for multiple comparisons across items by default (BH by default, Bonferroni also available) before the flagging rule is applied.

For PCM, RSM and MFRM, which have a threshold/category structure in addition to item location, `dif_test()` also separately tests for Differential Category Functioning (DCF) via `threshold_dif_table`. Here we make a distinction between DCF and the more standard DSF approach, which tests for differences in threshold locations: instead, DCF tests for differences in category *widths* between adjacent thresholds ($ \tau_{k+1} - \tau_{k} $), tested category by category, reference vs. focal. This matters because two groups can show identical overall item difficulty — nothing flagged in `dif_table` — while still using the rating scale differently, e.g. one group's raters compress two middle categories together or all but skip one; that is a real, substantive difference in how the item functions, but it is invisible to any test of the item as a whole. The issue with DSF compared to DCF is that thresholds are interdependent; with a maximum score $ m $, there are $ m $ thresholds but only $ m - 1 $ degrees of freedom, due to the sum-to-zero constraint on Rasch-Andrich thresholds. For example, with a maximum score of 2, if one threshold shifts, the other must by definition shift by an equal and opposite amount; the standard error of the middle (only unbounded) category is precisely twice that of each threshold (equal by definition). This means that a change in the functioning (width) of one category propagates through the rest of the thresholds, confounding the DSF test. DCF, on the other hand, avoids this by examining the appropriate, independent unit (with the correct degrees of freedom: $ m - 1 $ df for $ m - 1 $ bounded categories). 

`dif_test()` also optionally adds an ETS-style DIF-magnitude `Category` column (`category=True`), following Zwick, Thayer & Mazzeo (1997): each item is classified 'A' (negligible), 'B+'/'B-' (slight to moderate), or 'C+'/'C-' (moderate to large). The sign gives the direction: '+' means the item is harder for the focal group than the reference group, '-' the converse. 'B' requires |Difference| ≥ 0.43 logits plus a significant Wald/Welch $ p $-value (`category_alpha=0.05` by default); 'C' requires |Difference| ≥ 0.64 logits plus a separate, one-sided test that the difference is significantly above the B boundary itself (P(|Difference| ≤ 0.43) < 0.05), not merely significantly nonzero. This classification is independent of the plain `Flagged` column (which uses `alpha`/`logit_threshold` directly) — it exists so results can also be reported in ETS categories. It is scoped to item-location DIF (`dif_table`) only: the 0.43/0.64 logit boundaries are calibrated in the literature for item-location DIF specifically, not for DCF category-width tests.

This notebook simulates two groups (A and B) with known, deliberately injected DIF, and checks whether `dif_test()` successfully detects it.

In [1]:
import numpy as np
import pandas as pd
from raschpy import SLM, PCM, RSM, MFRM
from raschpy.simulation import SLM_Sim, PCM_Sim, RSM_Sim
from raschpy.simulation.mfrm_sim import MFRM_Sim_Global, MFRM_Sim_Items, MFRM_Sim_Centrality

#### SLM

Simulate two groups of 300 persons each on the same 12 items. Group B has genuine DIF injected on three items: Item_1 (+1.0 logits, harder for B), Item_2 (-1.0 logits, easier for B), Item_3 (+0.5 logits). All other items are unaffected.

In [2]:
no_of_persons = 300
sim_a = SLM_Sim(no_of_items=12, no_of_persons=no_of_persons, item_range=2, seed=1)

items_b = sim_a.items.copy()
items_b['Item_1'] += 1.0
items_b['Item_2'] -= 1.0
items_b['Item_3'] += 0.5

sim_b = SLM_Sim(no_of_items=12, no_of_persons=no_of_persons, item_range=2,
                manual_items=items_b, manual_item_names=sim_a.item_names, seed=2)

resp_a, resp_b = sim_a.responses.copy(), sim_b.responses.copy()
resp_a.index = [f'A_{i}' for i in range(no_of_persons)]
resp_b.index = [f'B_{i}' for i in range(no_of_persons)]
responses = pd.concat([resp_a, resp_b])

exogenous = pd.DataFrame({'Group': ['A'] * no_of_persons + ['B'] * no_of_persons}, index=responses.index)

slm = SLM(responses, exogenous=exogenous)

Run the DIF test, comparing group B against reference group A:

In [3]:
slm.dif_test('Group',
             reference='A',
             selection_method='wald',   # purification method: 'wald' (default) | 'robust_z' | 'none'
             test='both',               # per-item flagging: 'wald' | 'lr' | 'both'
             omnibus=True,              # Andersen-style joint test, all items at once
             category=True,             # adds an ETS-style A/B/C DIF-magnitude category column
             correction='bh',           # multiple-comparison correction: 'bh' | 'bonferroni' | None
             welch=True,                # Welch's t-test (Satterthwaite df) instead of z, more conservative for small/unequal groups
             no_of_samples=500)

round(slm.dif_table, 3)

,Group,Reference,Focal,Focal (purified),Difference,SE,z,p,p (corrected),Selected,Flagged,df,Category,LR,p_LR,p_LR (corrected),Flagged_LR
Item,,,,,,,,,,,,,,,,,
Item_1,B,0.190,1.057,1.105,0.915,0.217,4.220,0.000,0.000,False,True,529.078,C+,19.981,0.000,0.000,True
Item_2,B,0.894,-0.153,-0.104,-0.998,0.208,-4.793,0.000,0.000,False,True,537.707,C-,22.890,0.000,0.000,True
Item_3,B,-0.197,0.389,0.438,0.634,0.208,3.043,0.002,0.010,False,True,536.809,B+,11.988,0.001,0.002,True
Item_4,B,0.590,0.279,0.294,-0.296,0.212,-1.392,0.164,0.493,True,False,537.967,A,2.266,0.132,0.397,False
Item_5,B,-1.226,-1.036,-0.962,0.264,0.228,1.159,0.247,0.592,True,False,538.000,A,1.006,0.316,0.640,False
Item_6,B,-0.303,-0.382,-0.376,-0.074,0.212,-0.347,0.729,0.777,True,False,537.825,A,0.182,0.670,0.789,False
Item_7,B,0.075,0.078,0.133,0.058,0.205,0.283,0.777,0.777,True,False,537.844,A,0.075,0.784,0.789,False
Item_8,B,-0.026,-0.095,-0.089,-0.063,0.211,-0.296,0.768,0.777,True,False,537.310,A,0.072,0.789,0.789,False
Item_9,B,-0.850,-0.773,-0.710,0.140,0.215,0.652,0.515,0.777,True,False,537.999,A,0.320,0.571,0.789,False


Items 1-3, where DIF was injected, should be flagged with `Difference` estimates close to the specified 1.0 / -1.0 / 0.5 logit shifts. The remaining items should not be flagged.

In [4]:
slm.dif_omnibus_table  # is there DIF at all, across every item jointly

,LR,df,p,Flagged
B,59.502164,11,0.0,True


#### PCM

Simulate two groups on 15 4-category items (maximum score 3). Group B has the same item-location DIF pattern as before (Item_1/2/3), plus a pure category-width (DCF) shift on Item_4 only — its overall item location is left unchanged, but the width of Category 1 is increased by 0.8 logits, so it should be flagged in `threshold_dif_table` rather than `dif_table`.

In [5]:
no_of_persons = 1000
max_score_vector = [3] * 15
sim_a = PCM_Sim(no_of_items=15, no_of_persons=no_of_persons, max_score_vector=max_score_vector, item_range=2, seed=1)

items_b = sim_a.items.copy()
items_b['Item_1'] += 1.0
items_b['Item_2'] -= 1.0
items_b['Item_3'] += 0.5

# Pin thresholds to sim_a's own for every item except Item_4, which gets a
# deliberate category-width (DCF) shift with no item-location shift.
manual_thresholds = []
for item in sim_a.item_names:
    vals = sim_a.thresholds.loc[item].dropna().values.copy()
    if item == 'Item_4':
        # Isolate the shift to a single category width (w1): subtracting a
        # constant from every threshold preserves all pairwise differences, so
        # perturbing tau_1 alone and re-centering shifts w1 without touching w2
        # (unlike recomputing just the last threshold, which would couple them).
        vals[0] -= 0.8
        vals -= vals.mean()
    else:
        vals[-1] = -vals[:-1].sum()  # exact zero-sum, avoiding float residue
    manual_thresholds.append(vals)

sim_b = PCM_Sim(no_of_items=15, no_of_persons=no_of_persons, max_score_vector=max_score_vector, item_range=2,
                manual_items=items_b, manual_item_names=sim_a.item_names,
                manual_thresholds=manual_thresholds, seed=2)

resp_a, resp_b = sim_a.responses.copy(), sim_b.responses.copy()
resp_a.index = [f'A_{i}' for i in range(no_of_persons)]
resp_b.index = [f'B_{i}' for i in range(no_of_persons)]
responses = pd.concat([resp_a, resp_b])
exogenous = pd.DataFrame({'Group': ['A'] * no_of_persons + ['B'] * no_of_persons}, index=responses.index)

pcm = PCM(responses, max_score_vector=max_score_vector, exogenous=exogenous)

In [6]:
pcm.dif_test('Group',
             reference='A',
             selection_method='wald',
             test='both',
             omnibus=True,
             omnibus_scope='full',
             category=True,
             correction='bh',
             welch=True,
             no_of_samples=300)

round(pcm.dif_table, 3)   # item-location DIF

,Group,Reference,Focal,Focal (purified),Difference,SE,z,p,p (corrected),Selected,Flagged,df,Category,LR,p_LR,p_LR (corrected),Flagged_LR
Item,,,,,,,,,,,,,,,,,
Item_1,B,-0.959,0.044,0.101,1.060,0.125,8.476,0.000,0.000,False,True,1952.378,C+,159.067,0.000,0.000,True
Item_2,B,-0.251,-1.226,-1.168,-0.917,0.087,-10.542,0.000,0.000,False,True,1944.226,C-,169.232,0.000,0.000,True
Item_3,B,0.727,1.126,1.183,0.457,0.104,4.373,0.000,0.000,False,True,1982.384,B+,43.569,0.000,0.000,True
Item_4,B,-0.609,-0.582,-0.518,0.091,0.098,0.934,0.350,0.657,True,False,1971.956,A,1.190,0.275,0.535,False
Item_5,B,-0.067,-0.240,-0.191,-0.123,0.104,-1.184,0.236,0.525,True,False,1995.650,A,2.270,0.132,0.330,False
Item_6,B,0.983,0.949,1.009,0.026,0.082,0.315,0.753,0.921,True,False,1993.541,A,0.073,0.786,0.976,False
Item_7,B,-0.281,-0.179,-0.121,0.160,0.086,1.862,0.063,0.235,False,False,1990.543,A,4.504,0.034,0.127,False
Item_8,B,-0.300,-0.383,-0.327,-0.027,0.112,-0.240,0.810,0.921,True,False,1993.585,A,0.081,0.776,0.976,False
Item_9,B,1.065,0.998,1.055,-0.010,0.105,-0.099,0.921,0.921,True,False,1995.811,A,0.029,0.866,0.976,False


In [7]:
round(pcm.threshold_dif_table, 3)   # per-item category-width (DCF) test

Group  Reference  Focal  Difference     SE      z        df  \
Item    Category                                                               
Item_1  1            B      4.205  3.914      -0.291  0.336 -0.865  1819.222   
        2            B      2.909  2.994       0.084  0.249  0.338  1871.072   
Item_2  1            B      1.996  1.390      -0.606  0.277 -2.193  1844.549   
        2            B      2.010  2.356       0.346  0.208  1.661  1993.478   
Item_3  1            B      1.331  1.438       0.107  0.211  0.508  1989.333   
        2            B      3.759  3.466      -0.293  0.310 -0.945  1964.938   
Item_4  1            B      1.511  2.505       0.994  0.289  3.441  1992.831   
        2            B      3.468  3.466      -0.002  0.214 -0.008  1994.562   
Item_5  1            B      2.348  2.577       0.229  0.270  0.848  1992.473   
        2            B      3.961  3.750      -0.211  0.249 -0.848  1994.024   
Item_6  1            B      1.132  1.662       0.530  0.209  2.539  1995.152   
        2            B      1.863  1.400      -0.463  0.273 -1.694  1989.271   
Item_7  1            B      1.726  1.317      -0.410  0.250 -1.641  1985.503   
        2            B      3.188  3.246       0.058  0.218  0.264  1993.965   
Item_8  1            B      3.746  3.732      -0.013  0.295 -0.046  1995.485   
        2            B      3.193  3.006      -0.187  0.239 -0.785  1988.825   
Item_9  1            B      1.769  1.828       0.059  0.203  0.289  1995.644   
        2            B      3.250  3.515       0.264  0.316  0.836  1995.983   
Item_10 1            B      3.110  2.939      -0.171  0.293 -0.583  1994.446   
        2            B      1.982  2.151       0.169  0.203  0.833  1994.426   
Item_11 1            B      0.553  0.978       0.425  0.230  1.847  1995.897   
        2            B      1.093  0.676      -0.417  0.280 -1.489  1994.420   
Item_12 1            B      2.975  2.738      -0.238  0.231 -1.029  1992.181   
        2            B      0.817  0.583      -0.233  0.226 -1.033  1993.082   
Item_13 1            B      2.554  2.129      -0.424  0.283 -1.497  1992.952   
        2            B      3.084  2.907      -0.177  0.212 -0.836  1995.950   
Item_14 1            B      2.621  2.481      -0.140  0.224 -0.625  1995.833   
        2            B      2.545  2.476      -0.070  0.231 -0.302  1992.325   
Item_15 1            B      0.887  0.771      -0.116  0.239 -0.483  1993.605   
        2            B      0.452  0.455       0.003  0.248  0.012  1993.834   

                      p  p (corrected)  Flagged  
Item    Category                                 
Item_1  1         0.387          0.735    False  
        2         0.735          0.735    False  
Item_2  1         0.028          0.057    False  
        2         0.097          0.097    False  
Item_3  1         0.611          0.611    False  
        2         0.345          0.611    False  
Item_4  1         0.001          0.001     True  
        2         0.994          0.994    False  
Item_5  1         0.397          0.397    False  
        2         0.396          0.397    False  
Item_6  1         0.011          0.022     True  
        2         0.090          0.090    False  
Item_7  1         0.101          0.202    False  
        2         0.791          0.791    False  
Item_8  1         0.964          0.964    False  
        2         0.432          0.865    False  
Item_9  1         0.773          0.773    False  
        2         0.403          0.773    False  
Item_10 1         0.560          0.560    False  
        2         0.405          0.560    False  
Item_11 1         0.065          0.130    False  
        2         0.137          0.137    False  
Item_12 1         0.304          0.304    False  
        2         0.302          0.304    False  
Item_13 1         0.135          0.269    False  
        2         0.403          0.403    False  
Item_14 1         0.532          0.763    False  
      

Item_4 should be flagged in `threshold_dif_table` (its category structure differs between groups) but not in `dif_table` (its item location is unaffected) — the two phenomena are deliberately kept independent in this simulation so the signal is unambiguous.

#### RSM

Simulate two groups on 20 items sharing one rating scale with 4 categories (maximum score 3). Group B again gets the Item_1/2/3 location DIF, plus the same +0.8 shift to one shared category width used for PCM's Item_4 — since RSM's thresholds (and therefore widths) are shared across all items, this is a uniform instrument-wide DCF effect rather than a single-item one.

In [8]:
no_of_persons = 1000
sim_a = RSM_Sim(no_of_items=20, no_of_persons=no_of_persons, max_score=3, item_range=2, seed=1)

items_b = sim_a.items.copy()
items_b['Item_1'] += 1.0
items_b['Item_2'] -= 1.0
items_b['Item_3'] += 0.5

thresholds_b = sim_a.thresholds.copy()
thresholds_b[1] -= 0.8
thresholds_b -= thresholds_b.mean()

sim_b = RSM_Sim(no_of_items=20, no_of_persons=no_of_persons, max_score=3, item_range=2,
                manual_items=items_b, manual_item_names=sim_a.item_names,
                manual_thresholds=thresholds_b, seed=2)

resp_a, resp_b = sim_a.responses.copy(), sim_b.responses.copy()
resp_a.index = [f'A_{i}' for i in range(no_of_persons)]
resp_b.index = [f'B_{i}' for i in range(no_of_persons)]
responses = pd.concat([resp_a, resp_b])
exogenous = pd.DataFrame({'Group': ['A'] * no_of_persons + ['B'] * no_of_persons}, index=responses.index)

rsm = RSM(responses, exogenous=exogenous)

In [9]:
rsm.dif_test('Group',
             reference='A',
             selection_method='robust_z',
             test='both',
             omnibus=True,
             category=True,
             correction='bh',
             welch=True,
             no_of_samples=300)

round(rsm.dif_table, 3)   # item-location DIF

,Group,Reference,Focal,Focal (purified),Difference,SE,z,p,p (corrected),Selected,Flagged,df,Category,LR,p_LR,p_LR (corrected),Flagged_LR
Item,,,,,,,,,,,,,,,,,
Item_1,B,-0.908,-0.048,-0.032,0.876,0.081,10.785,0.000,0.000,False,True,1991.813,C+,126.301,0.000,0.000,True
Item_2,B,-0.381,-1.371,-1.354,-0.974,0.080,-12.180,0.000,0.000,False,True,1995.213,C-,157.530,0.000,0.000,True
Item_3,B,0.665,1.078,1.095,0.430,0.090,4.776,0.000,0.000,False,False,1940.386,A,22.487,0.000,0.000,False
Item_4,B,-0.700,-0.741,-0.726,-0.025,0.080,-0.313,0.754,0.975,True,False,1995.111,A,0.149,0.700,0.895,False
Item_5,B,-0.287,-0.171,-0.146,0.141,0.079,1.794,0.073,0.243,True,False,1997.483,A,3.172,0.075,0.250,False
Item_6,B,0.791,0.780,0.796,0.006,0.089,0.066,0.947,0.975,True,False,1981.349,A,0.004,0.950,0.950,False
Item_7,B,-0.313,-0.374,-0.360,-0.047,0.080,-0.589,0.556,0.914,True,False,1981.336,A,0.315,0.574,0.895,False
Item_8,B,-0.489,-0.545,-0.531,-0.042,0.079,-0.533,0.594,0.914,True,False,1986.278,A,0.051,0.821,0.950,False
Item_9,B,0.967,0.933,0.948,-0.019,0.093,-0.200,0.841,0.975,True,False,1966.063,A,0.210,0.647,0.895,False


In [10]:
rsm.dif_omnibus_table     # joint item-location test

,LR,df,p,Flagged
B,301.957358,19,0.0,True


In [11]:
round(rsm.threshold_dif_table, 3)   # shared category-width (DCF) DIF

,Group,Category,Reference,Focal,Difference,SE,z,p,p (corrected),Flagged,df
0,B,1,3.887,4.788,0.901,0.077,11.771,0.00,0.00,True,1951.773
1,B,2,1.555,1.508,-0.048,0.064,-0.739,0.46,0.46,False,1996.258


#### MFRM

Simulate two groups of 300 persons each, rated by the same 6 raters on 10 items (maximum score 4), under the 'global' rater model. Group B has the same item-location DIF injected on Items 1-3 as the other model classes, plus a genuine category-width (DCF) shift on the shared threshold vector -- raters are held fixed between groups (same severities, fresh simulated ratings):

In [12]:
no_of_persons = 300
sim_a = MFRM_Sim_Global(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=6, max_score=4,
                         manual_person_names=[f'A_{i}' for i in range(no_of_persons)], seed=1)

items_b = sim_a.items.copy()
items_b['Item_1'] += 1.0
items_b['Item_2'] -= 1.0
items_b['Item_3'] += 0.5

thresholds_b = sim_a.thresholds.copy()
thresholds_b[1] -= 0.8
thresholds_b -= thresholds_b.mean()

sim_b = MFRM_Sim_Global(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=6, max_score=4,
                         manual_items=items_b, manual_item_names=sim_a.item_names,
                         manual_thresholds=thresholds_b.values,
                         manual_facet_effects=sim_a.facet_effects, manual_facet_names=sim_a.facet_names,
                         manual_person_names=[f'B_{i}' for i in range(no_of_persons)], seed=2)

responses = pd.concat([sim_a.responses, sim_b.responses])
exogenous = pd.DataFrame({'Group': ['A'] * no_of_persons + ['B'] * no_of_persons},
                          index=[f'A_{i}' for i in range(no_of_persons)] + [f'B_{i}' for i in range(no_of_persons)])

mfrm = MFRM(responses, max_score=4, exogenous=exogenous)

`model=` selects which of the 8 rater representations each group is calibrated under -- item-location DIF and DCF are conceptually model-independent, so this only affects each group's internal `calibrate()` call. Run the DIF/DCF test, comparing group B against reference group A:

In [13]:
mfrm.dif_test('Group',
              model='global',
              reference='A',
              selection_method='wald',
              test='both',
              omnibus=True,
              category=True,
              correction='bh',
              welch=True,
              no_of_samples=300,
              seed=1)

round(mfrm.dif_table, 3)   # item-location DIF

/Users/markelliott/Documents/GitHub/RaschPy/raschpy/base.py:1951: UserWarning: Wald-based anchor selection reached min_anchors=6 before every remaining item cleared alpha=0.05. Residual outliers may remain in the selected anchor set.
  ar = self._wald_anchor_selection(
/Users/markelliott/Documents/GitHub/RaschPy/raschpy/base.py:1951: UserWarning: Wald-based anchor selection reached min_anchors=6 before every remaining item cleared alpha=0.05. Residual outliers may remain in the selected anchor set.
  ar = self._wald_anchor_selection(
/Users/markelliott/Documents/GitHub/RaschPy/raschpy/base.py:1951: UserWarning: Wald-based anchor selection reached min_anchors=6 before every remaining item cleared alpha=0.05. Residual outliers may remain in the selected anchor set.
  ar = self._wald_anchor_selection(
/Users/markelliott/Documents/GitHub/RaschPy/raschpy/base.py:1951: UserWarning: Wald-based anchor selection reached min_anchors=6 before every remaining item cleared alpha=0.05. Residual outl

,Group,Reference,Focal,Focal (purified),Difference,SE,z,p,p (corrected),Selected,Flagged,df,Category,LR,p_LR,p_LR (corrected),Flagged_LR
Item,,,,,,,,,,,,,,,,,
Item_1,B,0.152,1.115,1.172,1.020,0.067,15.226,0.000,0.000,False,True,582.917,C+,273.787,0.000,0.000,True
Item_2,B,1.009,-0.052,0.005,-1.004,0.061,-16.557,0.000,0.000,False,True,596.356,C-,275.441,0.000,0.000,True
Item_3,B,-0.082,0.306,0.363,0.444,0.064,6.917,0.000,0.000,False,True,592.805,B+,55.086,0.000,0.000,True
Item_4,B,0.657,0.726,0.783,0.126,0.065,1.929,0.054,0.135,False,False,597.725,A,4.453,0.035,0.087,False
Item_5,B,-1.047,-1.041,-0.993,0.055,0.061,0.897,0.370,0.617,True,False,594.493,A,0.947,0.330,0.551,False
Item_6,B,-0.284,-0.334,-0.296,-0.013,0.065,-0.196,0.845,0.859,True,False,597.931,A,0.040,0.841,0.841,False
Item_7,B,0.083,0.034,0.072,-0.011,0.061,-0.178,0.859,0.859,True,False,588.038,A,0.048,0.826,0.841,False
Item_8,B,0.256,0.127,0.154,-0.102,0.063,-1.623,0.105,0.210,True,False,595.788,A,3.087,0.079,0.158,False
Item_9,B,-0.530,-0.600,-0.565,-0.035,0.060,-0.582,0.561,0.783,True,False,597.962,A,0.309,0.578,0.810,False


In [14]:
mfrm.dif_omnibus_table   # item-isolated omnibus test (facet_effects pinned at the pooled value)

,LR,df,p,Flagged
B,609.238531,9,0.0,True


In [15]:
round(mfrm.threshold_dif_table, 3)   # DCF -- shared category-width shift, same rationale as RSM's own DCF

,Group,Category,Reference,Focal,Difference,SE,z,p,p (corrected),Flagged,df
0,B,1,3.800,4.552,0.753,0.116,6.487,0.000,0.000,True,596.916
1,B,2,2.543,2.595,0.052,0.058,0.895,0.371,0.371,False,596.055
2,B,3,2.436,2.323,-0.113,0.107,-1.058,0.290,0.371,False,597.580


Items 1-3, where item-location DIF was injected, should be flagged with `Difference` estimates close to the specified shifts. In `threshold_dif_table`, Category 1 (the width between thresholds 1 and 2) should be flagged, where the DCF shift was injected.

Item_3 is a useful edge case: its injected DIF (0.5 logits) is only marginally above the ETS Category B threshold of 0.43. In this simulated sample, data noise pulls the observed `Difference` down to 0.406 -- decisively significant (`p` = 0.000) but just under the magnitude gate, so it is correctly *not* flagged. Raising the injected value to 0.55 logits recovers an observed `Difference` of 0.439, just clearing the threshold, and Item_3 is then flagged cleanly (Category 'B+'). Neither result is a test failure -- both are the effect-size gate doing exactly what it is meant to do, screening out an effect that is statistically real but sits right on the boundary of practical significance.

`drf_test()` is the DIF analogue applied to rater severity instead of item location. Its test shape depends on `model` -- three genuinely different shapes, not one design stretched across all 8 rater representations: a single Wald test per bridging rater for `'global'`; one per (rater, item/threshold) pair for the vector/matrix models (`'items'`/`'thresholds'`/`'bivector'`/`'matrix'`); and one joint, covariance-aware test over a rater's raw stretch parameters for the stretch models (`'centrality'`/`'pseudo_halo'`/`'bistretch'`). The mechanics within each group are the same regardless of which specific model is used, so one worked example per shape below covers all 8.

**`'global'`**: a single scalar severity per rater. Simulate a fresh pair of groups with items held fixed between groups this time, and a genuine severity shift injected on Rater_1:

In [16]:
sim_a = MFRM_Sim_Global(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=6, max_score=4,
                         manual_person_names=[f'A_{i}' for i in range(no_of_persons)], seed=1)

facets_b = sim_a.facet_effects.copy()
facets_b['Rater_1'] += 1.2

sim_b = MFRM_Sim_Global(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=6, max_score=4,
                         manual_items=sim_a.items, manual_item_names=sim_a.item_names,
                         manual_facet_effects=facets_b, manual_facet_names=sim_a.facet_names,
                         manual_person_names=[f'B_{i}' for i in range(no_of_persons)], seed=2)

responses = pd.concat([sim_a.responses, sim_b.responses])
mfrm = MFRM(responses, max_score=4, exogenous=exogenous)

mfrm.drf_test('Group',
              model='global',
              reference='A',
              selection_method='wald',
              welch=True,
              no_of_samples=300,
              seed=1)

round(mfrm.drf_table, 3)   # per-rater DRF, restricted to raters present in both groups

,Group,Reference,Focal,Focal (purified),Difference,SE,z,p,p (corrected),Selected,Flagged,df
Rater,,,,,,,,,,,,
Rater_1,B,-0.453,0.571,0.820,1.274,0.054,23.482,0.000,0.000,False,True,596.575
Rater_2,B,-0.027,-0.293,-0.074,-0.048,0.050,-0.950,0.343,0.343,True,False,593.925
Rater_3,B,-0.158,-0.428,-0.211,-0.053,0.052,-1.019,0.309,0.343,True,False,578.714
Rater_4,B,1.245,1.074,1.324,0.079,0.052,1.502,0.134,0.267,False,False,591.609
Rater_5,B,-0.760,-0.860,-0.610,0.150,0.048,3.102,0.002,0.006,False,False,597.775
Rater_6,B,0.153,-0.063,0.205,0.052,0.050,1.047,0.296,0.343,True,False,597.418


Rater_1, where a severity shift was injected, should be flagged; the other raters should not.

**`'items'`** (representative of the vector/matrix shapes): one facet_effect per (rater, item) pair, tested individually. Inject a shift on a single (rater, item) cell -- Rater_1's own effect on Item_1 -- with every other cell, and every item location, held fixed:

In [17]:
sim_a = MFRM_Sim_Items(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=6, max_score=4,
                        manual_person_names=[f'A_{i}' for i in range(no_of_persons)], seed=1)

facets_b = sim_a.facet_effects.copy()   # (rater x item) DataFrame
facets_b.loc['Rater_1', 'Item_1'] += 1.2

sim_b = MFRM_Sim_Items(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=6, max_score=4,
                        manual_items=sim_a.items, manual_item_names=sim_a.item_names,
                        manual_facet_effects=facets_b, manual_facet_names=sim_a.facet_names,
                        manual_person_names=[f'B_{i}' for i in range(no_of_persons)], seed=2)

responses = pd.concat([sim_a.responses, sim_b.responses])
mfrm = MFRM(responses, max_score=4, exogenous=exogenous)

mfrm.drf_test('Group',
              model='items',
              reference='A',
              selection_method='wald',
              welch=True,
              no_of_samples=300,
              seed=1)

round(mfrm.drf_table, 3).head(12)   # one row per (rater, item) pair -- 'thresholds'/'bivector'/'matrix' add an item/threshold column the same way

,Group,Item,Reference,Focal,Focal (purified),Difference,SE,z,p,Selected,df,p (corrected),Flagged
Rater,,,,,,,,,,,,,
Rater_1,B,Item_1,-0.639,0.296,0.525,1.164,0.162,7.172,0.000,False,573.766,0.000,True
Rater_2,B,Item_1,-0.520,-0.519,-0.290,0.230,0.155,1.487,0.138,False,591.222,0.688,False
Rater_3,B,Item_1,0.761,0.520,0.695,-0.066,0.149,-0.444,0.657,True,593.803,0.980,False
Rater_4,B,Item_1,0.534,0.351,0.599,0.065,0.162,0.401,0.689,True,588.843,0.980,False
Rater_5,B,Item_1,-0.592,-0.878,-0.708,-0.115,0.158,-0.727,0.467,True,595.448,0.930,False
Rater_6,B,Item_1,0.456,0.231,0.411,-0.046,0.150,-0.303,0.762,True,594.825,0.980,False
Rater_1,B,Item_2,-0.086,-0.166,-0.224,-0.137,0.150,-0.918,0.359,True,597.852,0.904,False
Rater_2,B,Item_2,-0.086,-0.254,-0.286,-0.200,0.157,-1.270,0.205,True,590.351,0.782,False
Rater_3,B,Item_2,-0.257,-0.306,-0.363,-0.106,0.151,-0.703,0.482,True,579.136,0.930,False


Only Rater_1's row for Item_1 should be flagged; every other (rater, item) pair should not.

**`'centrality'`** (representative of the stretch shapes): each rater carries two raw parameters, a severity shift ($\lambda_r$) and a threshold-stretch factor ($\omega_r$), from one shared regression -- tested jointly (`T`, a Mahalanobis-style statistic), not as two independent tests. Inject a shift on Rater_1's $\lambda_r$ only, leaving $\omega_r$ and every item/threshold unchanged:

In [18]:
sim_a = MFRM_Sim_Centrality(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=6, max_score=4,
                             manual_person_names=[f'A_{i}' for i in range(no_of_persons)], seed=1)

lambda_b = sim_a.lambda_.values.copy()
lambda_b[0] += 1.2   # Rater_1

sim_b = MFRM_Sim_Centrality(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=6, max_score=4,
                             manual_items=sim_a.items, manual_item_names=sim_a.item_names,
                             manual_thresholds=sim_a.thresholds.values,
                             manual_lambda=lambda_b, manual_omega=sim_a.omega.values,
                             manual_person_names=[f'B_{i}' for i in range(no_of_persons)], seed=2)

responses = pd.concat([sim_a.responses, sim_b.responses])
mfrm = MFRM(responses, max_score=4, exogenous=exogenous)

mfrm.drf_test('Group',
              model='centrality',
              reference='A',
              selection_method='wald',
              welch=True,
              no_of_samples=300,
              seed=1)

round(mfrm.drf_table[['Reference_lambda', 'Focal_lambda', 'Difference_lambda',
                      'Reference_omega', 'Focal_omega', 'Difference_omega', 'T', 'p', 'Flagged']], 3)

/Users/markelliott/Documents/GitHub/RaschPy/raschpy/mfrm.py:11096: RuntimeWarning: 9 of 300 bootstrap resamples raised the following warning (first occurrence shown; details such as item names and numbers may differ between resamples):
overflow encountered in exp
  ref_model.std_errors(model=model, **se_kw)


,Reference_lambda,Focal_lambda,Difference_lambda,Reference_omega,Focal_omega,Difference_omega,T,p,Flagged
Rater,,,,,,,,,
Rater_1,-0.175,1.121,1.296,1.390,1.383,-0.007,335.741,0.000,True
Rater_2,1.054,1.049,-0.005,0.912,0.917,0.005,0.059,0.971,False
Rater_3,-0.928,-1.002,-0.073,1.035,1.069,0.034,1.854,0.396,False
Rater_4,0.976,1.052,0.076,0.350,0.328,-0.022,3.625,0.163,False
Rater_5,-0.597,-0.440,0.157,1.285,1.244,-0.041,6.807,0.033,False
Rater_6,-0.330,-0.337,-0.007,1.028,1.059,0.031,1.349,0.510,False


Rater_1 should be flagged on the primary joint test (`Flagged`, driven by `T`) with `Difference_lambda` close to the injected shift and `Difference_omega` close to zero; `'pseudo_halo'`/`'bistretch'` follow the same joint-test shape, differing only in which raw parameters they carry.

#### Groups scored by different raters, or responding to different items


Groups are often scored by partly overlapping sets of raters, or respond to partly overlapping sets of items. `dif_test()` handles both: each group is calibrated on the items its persons responded to, only items shared by both groups are compared, and, for `MFRM`, each group is anchored on the facet elements shared by both groups before item locations and thresholds are compared.

**`MFRM`, partly overlapping raters**: group A is rated by Rater_1-6 and group B by Rater_3-8, under the `'items'` model (each rater has its own effect on each item). DIF is injected on Item_1 only:


In [19]:
sim_a = MFRM_Sim_Items(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=8, max_score=4,
                        manual_person_names=[f'A_{i}' for i in range(no_of_persons)], seed=1)

items_b = sim_a.items.copy()
items_b['Item_1'] += 1.0

sim_b = MFRM_Sim_Items(no_of_items=10, no_of_persons=no_of_persons, no_of_facet_elements=8, max_score=4,
                        manual_items=items_b, manual_item_names=sim_a.item_names,
                        manual_facet_effects=sim_a.facet_effects, manual_facet_names=sim_a.facet_names,
                        manual_person_names=[f'B_{i}' for i in range(no_of_persons)], seed=2)

sim_a.drop_raters(['Rater_7', 'Rater_8'])   # group A: Rater_1-6
sim_b.drop_raters(['Rater_1', 'Rater_2'])   # group B: Rater_3-8

responses = pd.concat([sim_a.responses, sim_b.responses])
persons = responses.index.get_level_values(1).unique()
exogenous = pd.DataFrame({'Group': [p[0] for p in persons]}, index=persons)

mfrm = MFRM(responses, max_score=4, exogenous=exogenous)
mfrm.dif_test('Group',
              model='items',
              reference='A',
              no_of_samples=300,
              seed=1)

round(mfrm.dif_table, 3)

/var/folders/hs/f4mv8_3d65bf5fln8jlxhxpc0000gn/T/ipykernel_18892/1362870342.py:12: UserWarning: 2 facet elements (e.g. raters) removed: ['Rater_7', 'Rater_8'].
  sim_a.drop_raters(['Rater_7', 'Rater_8'])   # group A: Rater_1-6
/var/folders/hs/f4mv8_3d65bf5fln8jlxhxpc0000gn/T/ipykernel_18892/1362870342.py:13: UserWarning: 2 facet elements (e.g. raters) removed: ['Rater_1', 'Rater_2'].
  sim_b.drop_raters(['Rater_1', 'Rater_2'])   # group B: Rater_3-8


/Users/markelliott/Documents/GitHub/RaschPy/raschpy/base.py:1951: UserWarning: Wald-based anchor selection reached min_anchors=6 before every remaining item cleared alpha=0.05. Residual outliers may remain in the selected anchor set.
  ar = self._wald_anchor_selection(


,Group,Reference,Focal,Focal (purified),Difference,SE,z,p,p (corrected),Selected,Flagged
Item,,,,,,,,,,,
Item_1,B,0.355,1.254,1.312,0.956,0.079,12.124,0.000,0.000,False,True
Item_2,B,1.004,0.964,1.054,0.050,0.073,0.692,0.489,0.599,True,False
Item_3,B,-0.206,-0.239,-0.146,0.060,0.069,0.871,0.384,0.599,True,False
Item_4,B,0.437,0.274,0.332,-0.105,0.071,-1.468,0.142,0.373,False,False
Item_5,B,-1.148,-1.227,-1.142,0.005,0.079,0.069,0.945,0.945,True,False
Item_6,B,-0.067,-0.278,-0.220,-0.153,0.079,-1.935,0.053,0.265,False,False
Item_7,B,0.147,0.100,0.189,0.042,0.068,0.614,0.539,0.599,True,False
Item_8,B,-0.016,-0.129,-0.062,-0.046,0.071,-0.654,0.513,0.599,True,False
Item_9,B,-0.234,-0.405,-0.347,-0.113,0.078,-1.442,0.149,0.373,False,False


Only Item_1 should be flagged. Each group's item locations are averages over the raters that scored it, so without anchoring on the raters shared by both groups (Rater_3-6), the differences between the two rater sets would show up as apparent DIF on the other items.

**`SLM`, partly overlapping items**: group A responded to Items 1-10 and group B to Items 3-12, with DIF injected on Item_5:


In [20]:
sim_a = SLM_Sim(no_of_items=12, no_of_persons=no_of_persons, item_range=2, seed=1)

items_b = sim_a.items.copy()
items_b['Item_5'] += 1.0

sim_b = SLM_Sim(no_of_items=12, no_of_persons=no_of_persons, item_range=2,
                manual_items=items_b, manual_item_names=sim_a.item_names, seed=2)

resp_a, resp_b = sim_a.responses.copy(), sim_b.responses.copy()
resp_a.index = [f'A_{i}' for i in range(no_of_persons)]
resp_b.index = [f'B_{i}' for i in range(no_of_persons)]
resp_a[['Item_11', 'Item_12']] = np.nan   # group A: Items 1-10
resp_b[['Item_1', 'Item_2']] = np.nan     # group B: Items 3-12
responses = pd.concat([resp_a, resp_b])

exogenous = pd.DataFrame({'Group': ['A'] * no_of_persons + ['B'] * no_of_persons}, index=responses.index)

slm = SLM(responses, exogenous=exogenous)
slm.dif_test('Group',
             reference='A')

round(slm.dif_table, 3)

/Users/markelliott/Documents/GitHub/RaschPy/raschpy/base.py:1951: UserWarning: Wald-based anchor selection reached min_anchors=6 before every remaining item cleared alpha=0.05. Residual outliers may remain in the selected anchor set.
  ar = self._wald_anchor_selection(
/Users/markelliott/Documents/GitHub/RaschPy/raschpy/base.py:1951: UserWarning: Wald-based anchor selection reached min_anchors=6 before every remaining item cleared alpha=0.05. Residual outliers may remain in the selected anchor set.
  ar = self._wald_anchor_selection(
/Users/markelliott/Documents/GitHub/RaschPy/raschpy/base.py:1951: UserWarning: Wald-based anchor selection reached min_anchors=6 before every remaining item cleared alpha=0.05. Residual outliers may remain in the selected anchor set.
  ar = self._wald_anchor_selection(


,Group,Reference,Focal,Focal (purified),Difference,SE,z,p,p (corrected),Selected,Flagged
Item,,,,,,,,,,,
Item_3,B,-0.080,-0.030,0.087,0.167,0.213,0.784,0.433,0.861,True,False
Item_4,B,0.717,0.352,0.400,-0.317,0.218,-1.456,0.145,0.581,False,False
Item_5,B,-1.106,-0.227,-0.179,0.927,0.224,4.144,0.000,0.000,False,True
Item_6,B,-0.184,-0.312,-0.224,-0.040,0.214,-0.189,0.850,0.861,True,False
Item_7,B,0.205,0.143,0.242,0.037,0.211,0.175,0.861,0.861,True,False
Item_8,B,0.096,-0.030,0.058,-0.038,0.212,-0.180,0.857,0.861,True,False
Item_9,B,-0.721,-0.722,-0.614,0.107,0.218,0.491,0.624,0.861,True,False
Item_10,B,-0.279,-0.300,-0.194,0.085,0.214,0.399,0.690,0.861,True,False


Only the items shared by both groups (Items 3-10) appear in `dif_table`, and only Item_5 should be flagged. With more than two groups, a focal group that shares no items with the reference group cannot be compared with it; it is left out with a `UserWarning` and recorded in `slm.dif_excluded_groups`.
